# ✨ IntelliCode-SL | Formatter SLM Fine-Tuning
**Model:** `meta-llama/Llama-3.2-1B-Instruct`  
**Task:** Takes raw pointwise agent output and rewrites it into a clean, well-structured, professional response.

**Why Llama-3.2-1B?**
- Already used in notebook 03 for explain/document tasks
- Strong natural language quality — formatting is a language task, not a code task
- Lightweight enough for fast inference as the final pipeline node

> ⚠️ Accept Meta's license first: [Llama-3.2-1B-Instruct on HuggingFace](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct)  
> Run cells top to bottom. GPU runtime required (Runtime → Change runtime type → T4 GPU)

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────
!pip install -q unsloth transformers datasets peft accelerate bitsandbytes trl
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────
import os, torch
from google.colab import drive
from huggingface_hub import login
from datasets import load_dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer
print("✅ Imports done | GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────
drive.mount("/content/drive")

DRIVE_BASE   = "/content/drive/MyDrive/IntelliCode-SL"
DATASET_PATH = f"{DRIVE_BASE}/datasets/formatter_dataset.json"
ADAPTER_SAVE = f"{DRIVE_BASE}/adapters/formatter_adapter"

os.makedirs(ADAPTER_SAVE, exist_ok=True)
print(f"✅ Drive mounted")
print(f"   Dataset path : {DATASET_PATH}")
print(f"   Adapter save : {ADAPTER_SAVE}")

In [ ]:
# ── Cell 4: HuggingFace Login ──────────────────────────────────
# ⚠️ Accept Meta license first: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"   # ← paste your token here
login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace")

In [ ]:
# ── Cell 5: Load Model via Unsloth ─────────────────────────────
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
print("✅ Model loaded")

In [ ]:
# ── Cell 6: Attach LoRA Adapter ────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
print("✅ LoRA adapter attached")
model.print_trainable_parameters()

## 📁 Dataset Format
Your `formatter_dataset.json` in Drive should look like:
```json
[
  {
    "input": "- Found bug on line 12: wrong operator used\n- Changed subtraction to addition\n- Added null check for safety",
    "output": "The bug was identified on line 12 where a subtraction operator was incorrectly used instead of addition. This has been corrected. A null check was also added to prevent potential errors when the input is empty."
  }
]
```
**Input:** Raw bullet-point style agent output (what your agents produce internally)  
**Output:** Clean, professional, well-structured response (what the user should see)

### Input types the formatter will handle:
- Debug agent output → structured explanation of what was fixed
- Generate agent output → clean code with brief explanation
- Modify agent output → description of changes made
- Explain agent output → clean readable explanation
- Document agent output → polished documentation

In [ ]:
# ── Cell 7: Load & Format Dataset ──────────────────────────────
PROMPT_TEMPLATE = """### Task:
You are a response formatter. Take the raw agent output below and rewrite it as a clean, professional, well-structured response that is easy for the user to read and understand.
Do not add new information. Only improve the clarity, structure, and readability of the existing content.

### Raw Agent Output:
{}

### Formatted Response:
{}"""

EOS = tokenizer.eos_token

def format_sample(sample):
    return {
        "text": PROMPT_TEMPLATE.format(
            sample["input"].strip(),
            sample["output"].strip()
        ) + EOS
    }

raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset     = raw_dataset.map(format_sample)

print(f"✅ Dataset loaded: {len(dataset)} samples")
print("\nSample preview:")
print(dataset[0]["text"][:400])

In [ ]:
# ── Cell 8: Training Arguments ─────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = "/content/formatter_checkpoints",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 20,
    save_strategy               = "epoch",
    warmup_ratio                = 0.03,
    lr_scheduler_type           = "cosine",
    report_to                   = "none",
)
print("✅ Training args set")

In [ ]:
# ── Cell 9: Train ──────────────────────────────────────────────
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    args               = training_args,
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

In [ ]:
# ── Cell 10: Save Adapter to Google Drive ──────────────────────
model.save_pretrained(ADAPTER_SAVE)
tokenizer.save_pretrained(ADAPTER_SAVE)
print(f"✅ Adapter saved to Drive → {ADAPTER_SAVE}")

In [ ]:
# ── Cell 11: Quick Inference Test ──────────────────────────────
FastLanguageModel.for_inference(model)

test_cases = [
    # Debug output
    """- bug found on line 8: used + instead of -
- also missing null check at start
- fixed both issues
- function now returns correct result""",

    # Generate output
    """- wrote function is_prime(n)
- handles edge cases: n < 2 returns False
- checks divisibility up to sqrt(n)
- returns True if no divisors found""",

    # Explain output
    """- function takes list as input
- iterates through each element
- multiplies each by 2
- returns new list with doubled values""",

    # Modify output
    """- added type hints to all parameters
- added docstring explaining purpose
- added input validation for negative numbers
- refactored loop to list comprehension
- added logging for debug purposes""",
]

for i, raw_output in enumerate(test_cases, 1):
    test_input = PROMPT_TEMPLATE.format(raw_output.strip(), "")
    inputs = tokenizer(test_input, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 200,
            temperature    = 0.3,
            do_sample      = True
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    formatted = result.split("### Formatted Response:")[-1].strip()
    print(f"{'='*55}")
    print(f"Test {i} - Raw Input:")
    print(raw_output.strip())
    print(f"\nFormatted Output:")
    print(formatted)
    print()